In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline , FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn_features.transformers import DataFrameSelector
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from imblearn.over_sampling import SMOTE


from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC # svm => SVC , SVR
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier


from sklearn import metrics
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import f1_score
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [5]:
#Read the data
df = pd.read_csv("D:\\deployment_mlops\\OS\\DVC\\data\\dataset.csv")
print(df.columns)  
df.head()


Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
df.shape

(10000, 14)

Drop unqiue Columns

In [7]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [8]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


detect the Outliers

In [ ]:
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
text_features = df.select_dtypes(include=[object]).columns.tolist()

C:\Users\USER\AppData\Local\Temp\ipykernel_26252\932531511.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_features = df.select_dtypes(include=[object]).columns.tolist()


In [10]:
numerical_features

['CreditScore',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited']

In [11]:
text_features

['Geography', 'Gender']

In [12]:
Q1 = df[numerical_features].quantile(0.25)
Q3=df[numerical_features].quantile(0.75)
IQR= Q3 - Q1
Lower_bound=Q1 - 1.5*IQR
upper_bound=Q3+ 1.5*IQR

In [13]:
Lower_bound

CreditScore           383.00000
Age                    14.00000
Tenure                 -3.00000
Balance           -191466.36000
NumOfProducts          -0.50000
HasCrCard              -1.50000
IsActiveMember         -1.50000
EstimatedSalary    -96577.09625
Exited                  0.00000
dtype: float64

In [14]:
upper_bound

CreditScore           919.00000
Age                    62.00000
Tenure                 13.00000
Balance            319110.60000
NumOfProducts           3.50000
HasCrCard               2.50000
IsActiveMember          2.50000
EstimatedSalary    296967.45375
Exited                  0.00000
dtype: float64

In [17]:
outliers = (
    (df[numerical_features] < Lower_bound) |
    (df[numerical_features] > upper_bound)
)

print(outliers.sum())

CreditScore          15
Age                 359
Tenure                0
Balance               0
NumOfProducts        60
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
Exited             2037
dtype: int64


In [12]:
df.shape

(10000, 11)

In [18]:
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

----------------------------

Now I should divide the data to Continuous Numerical Data 
and Binary Numerical Data & Target Column 

In [19]:
df.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')

In [20]:
numerical_features = [
    'CreditScore',
    'Age',
    'Balance',
    'EstimatedSalary'
]
binary_features = [
    'HasCrCard',
    'Tenure',
    'NumOfProducts',
    'IsActiveMember'
]
text_features = df.select_dtypes(include=[object]).columns.tolist()

target = 'Exited'


C:\Users\USER\AppData\Local\Temp\ipykernel_26252\275322322.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_features = df.select_dtypes(include=[object]).columns.tolist()


In [21]:
Q1 = df[numerical_features].quantile(0.25)
Q3 = df[numerical_features].quantile(0.75)

IQR = Q3 - Q1

Lower_bound = Q1 - 1.5 * IQR
Upper_bound = Q3 + 1.5 * IQR

In [22]:
outlier_mask = (
    (df[numerical_features] < Lower_bound) |
    (df[numerical_features] > Upper_bound)
)
print(outlier_mask.sum())

CreditScore         15
Age                359
Balance              0
EstimatedSalary      0
dtype: int64


In [23]:
print(outlier_mask.sum())

CreditScore         15
Age                359
Balance              0
EstimatedSalary      0
dtype: int64


In [24]:
for column in numerical_features:
    values = df.loc[outlier_mask[column], column]
    
    print(f"\n{column}:")
    print(values.to_list())


CreditScore:
[376, 376, 363, 359, 350, 350, 358, 351, 365, 367, 350, 350, 382, 373, 350]

Age:
[66, 75, 65, 73, 65, 72, 67, 67, 79, 80, 68, 75, 66, 66, 70, 63, 72, 64, 64, 70, 67, 82, 63, 69, 65, 69, 64, 65, 74, 67, 66, 67, 63, 70, 71, 72, 67, 74, 76, 66, 63, 66, 68, 67, 63, 71, 66, 69, 73, 65, 66, 64, 69, 64, 77, 74, 65, 70, 67, 69, 67, 74, 69, 74, 74, 64, 63, 63, 70, 74, 65, 72, 77, 66, 65, 74, 88, 63, 71, 63, 64, 67, 70, 68, 72, 71, 66, 75, 67, 73, 69, 76, 63, 85, 67, 74, 76, 66, 69, 66, 72, 63, 71, 63, 74, 67, 72, 72, 66, 84, 71, 66, 63, 74, 69, 84, 67, 64, 68, 66, 77, 70, 67, 79, 67, 76, 73, 66, 67, 64, 73, 76, 72, 64, 71, 63, 70, 65, 66, 65, 80, 66, 63, 63, 63, 63, 66, 74, 69, 63, 64, 76, 75, 68, 69, 77, 64, 66, 74, 71, 67, 68, 64, 68, 70, 64, 75, 66, 64, 78, 65, 74, 64, 64, 71, 77, 79, 70, 81, 64, 68, 68, 63, 79, 66, 64, 70, 69, 71, 72, 66, 68, 63, 71, 72, 72, 64, 78, 75, 65, 65, 67, 63, 68, 71, 73, 64, 66, 71, 69, 71, 66, 76, 69, 73, 64, 64, 75, 73, 71, 72, 63, 67, 68, 73, 67,

In [25]:
outlier_rows = df[outlier_mask.any(axis=1)]

print(outlier_rows)

      CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
7             376   Germany  Female   29       4  115046.74              4   
58            511     Spain  Female   66       4       0.00              1   
85            652     Spain  Female   75      10       0.00              2   
104           670     Spain  Female   65       1       0.00              1   
158           646    France  Female   73       6   97259.25              1   
...           ...       ...     ...  ...     ...        ...            ...   
9753          656   Germany    Male   68       7  153545.11              1   
9765          445    France    Male   64       2  136770.67              1   
9832          595   Germany  Female   64       2  105736.32              1   
9894          521    France  Female   77       6       0.00              2   
9936          609    France    Male   77       1       0.00              1   

      HasCrCard  IsActiveMember  EstimatedSalary  Exited  
7   

In [26]:
df['Age'].unique()

array([42, 41, 39, 43, 44, 50, 29, 27, 31, 24, 34, 25, 35, 45, 58, 32, 38,
       46, 36, 33, 40, 51, 61, 49, 37, 19, 66, 56, 26, 21, 55, 75, 22, 30,
       28, 65, 48, 52, 57, 73, 47, 54, 72, 20, 67, 79, 62, 53, 80, 59, 68,
       23, 60, 70, 63, 64, 18, 82, 69, 74, 71, 76, 77, 88, 85, 84, 78, 81,
       92, 83])

In [27]:
# Handling outliers using NumPy for each column
for col in df[numerical_features]:
    df[col] = np.where(df[col] < Lower_bound[col], Lower_bound[col],
                       np.where(df[col] > upper_bound[col],upper_bound[col],
                                df[col])
                       )		

In [28]:
outlier_mask = (
    (df[numerical_features] < Lower_bound) |
    (df[numerical_features] > Upper_bound)
)
print(outlier_mask.sum())

CreditScore        0
Age                0
Balance            0
EstimatedSalary    0
dtype: int64


Missing Data

In [29]:
df.isnull().sum().sum()

np.int64(0)

Check if the data is balanced 

In [30]:
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

Split the Data X , Y

In [31]:
X = df.drop(['Exited'], axis=1)
y = df['Exited']

Split Data Train Test 

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y)

Note : not forget to download 

uv add sklearn-features

In [33]:
Num_Pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='mean')),
    ('Scaler', StandardScaler())
])

Gender_Pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OrdinalEncoder())
])

Geography_Pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

Binary_Pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='most_frequent'))
])

all_pipeline = ColumnTransformer(
    transformers=[
        ('numerical', Num_Pipeline, numerical_features),
        ('gender', Gender_Pipeline, ['Gender']),
        ('geography', Geography_Pipeline, ['Geography']),
        ('binary', Binary_Pipeline, binary_features)
    ]
)




Apply this pipeline on X_train , X_test

In [34]:
X_train_final = all_pipeline.fit_transform(X_train)
X_test_final = all_pipeline.transform(X_test)

Imbalanced Data

In [35]:
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [36]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
classes

array([0, 1])

In [37]:
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)
weights

array([0.62794349, 2.45398773])

In [38]:
dict_weights = dict(zip(classes, weights))
dict_weights

{np.int64(0): np.float64(0.6279434850863422),
 np.int64(1): np.float64(2.4539877300613497)}

In [39]:
clf = RandomForestClassifier(
    class_weight='balanced'
)
LogisticRegression(class_weight='balanced')
DecisionTreeClassifier(class_weight='balanced')
SVC(class_weight='balanced')


,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


SMOTE

In [40]:
## 3. Using SMOTE for over sampling
#uv add imbalanced-learn
over = SMOTE(sampling_strategy=0.7)
X_train_resampled, y_train_resampled = over.fit_resample(X_train_final, y_train)

In [51]:
def train_model(X_train,y_train,X_test,y_test,plot_name='heatmap',class_weight=None):
    # Create model
    clf = RandomForestClassifier(n_estimators=500,max_depth=10,random_state=45,class_weight=class_weight)
    # Train
    clf.fit(X_train, y_train)
    # Predictions
    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)
    # Accuracy
    score_train = accuracy_score(y_train, y_pred_train)
    score_test = accuracy_score(y_test, y_pred_test)
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_test)

    plt.figure(figsize=(8, 6))

    sns.heatmap(cm,annot=True,cbar=False,fmt='d',cmap='Blues')

    plt.title(plot_name)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

    plt.savefig(
        f'{plot_name}.png',
        bbox_inches='tight',
        dpi=300
    )

    plt.close()

    # Save metrics
    with open('metrics.txt', 'a+') as f:
        f.write(f'{clf.__class__.__name__} - {plot_name}\n')
        f.write(f'Training Accuracy: {score_train * 100:.2f}%\n')
        f.write(f'Test Accuracy: {score_test * 100:.2f}%\n')
        f.write('-' * 40 + '\n')

    return clf, score_train, score_test


without imbalanced solution

In [52]:
clf1, train_acc_1, test_acc_1 = train_model(
    X_train_final,
    y_train,
    X_test_final,
    y_test,
    plot_name='without-imbalance'
)

Using Class weight

In [53]:
clf2, train_acc_2, test_acc_2 = train_model(
    X_train_final,
    y_train,
    X_test_final,
    y_test,
    plot_name='with-class-weights',
    class_weight='balanced'
)

using smote

In [54]:
clf3, train_acc_3, test_acc_3 = train_model(
    X_train_resampled,
    y_train_resampled,
    X_test_final,
    y_test,
    plot_name='with-SMOTE'
)

In [45]:
#Remove-Item -Recurse -Force .git